# Treinamento de modelos lineares
## Monitoria 3: da equação normal à regressão logística

**Pré-requisito:** Monitorias 1 e 2 (Python, pandas, matplotlib e scikit-learn) · **Ambiente:** Google Colab · **Duração:** 2h30–3h

**Base:** capítulo 4 (*Training Models*) do livro *Hands-On Machine Learning*, de Aurélien Géron ([notebooks no GitHub](https://github.com/ageron/handson-ml3)).

### Objetivo
Entender **como** um modelo linear aprende: o que significa "treinar", por que existem vários algoritmos de treino, como reconhecer *underfitting* e *overfitting* e como a regularização ajuda a controlar a complexidade.

**Pergunta-guia:** o que acontece por dentro quando chamamos `modelo.fit(X, y)`? E como escolher a complexidade de um modelo para que ele funcione bem com dados novos?

### Roteiro

| Bloco | Conteúdo | Tempo |
|---|---|---|
| 1 | Regressão linear e equação normal | 25 min |
| 2 | Gradiente descendente: batch, estocástico e mini-batch | 40 min |
| 3 | Regressão polinomial e curvas de aprendizado | 30 min |
| — | Intervalo | 10 min |
| 4 | Modelos regularizados: Ridge, Lasso, Elastic Net e *early stopping* | 35 min |
| 5 | Regressão logística e softmax | 30 min |
| 6 | Softmax "do zero" com NumPy (*opcional*) | casa |
| 7 | IA e fechamento | 10 min |

As seções marcadas como *opcional* podem ser feitas em casa. **Execute as células em ordem:** várias seções reaproveitam variáveis criadas antes.

## 0. Preparação

Todas as bibliotecas já estão instaladas no Colab. A célula abaixo também aumenta o tamanho padrão da fonte dos gráficos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import sklearn

np.random.seed(42)

plt.rc("font", size=14)
plt.rc("axes", labelsize=14, titlesize=14)
plt.rc("legend", fontsize=14)
plt.rc("xtick", labelsize=10)
plt.rc("ytick", labelsize=10)

print("scikit-learn:", sklearn.__version__)

## 1. Regressão linear e equação normal

Um **modelo linear** prevê somando as variáveis, cada uma multiplicada por um peso (parâmetro), mais um termo constante (*intercepto* ou *bias*):

$$\hat{y} = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \dots + \theta_n x_n$$

**Treinar** o modelo é encontrar os valores de $\theta$ que fazem as previsões ficarem o mais perto possível dos valores reais. A medida de "perto" mais comum é o **erro quadrático médio** (MSE, *mean squared error*):

$$\text{MSE}(\theta) = \frac{1}{m}\sum_{i=1}^{m}\left(\theta^\top \mathbf{x}^{(i)} - y^{(i)}\right)^2$$

Na regressão linear existe uma fórmula fechada que dá o $\theta$ que minimiza o MSE, a **equação normal**:

$$\hat{\theta} = \left(\mathbf{X}^\top \mathbf{X}\right)^{-1}\mathbf{X}^\top \mathbf{y}$$

### 1.1 Gerar dados sintéticos

Para enxergar o que o algoritmo faz, vamos criar dados em que **sabemos a resposta certa**: $y = 4 + 3x_1 + \text{ruído}$. O algoritmo não conhece o 4 nem o 3; o desafio é recuperá-los a partir dos dados.

In [ ]:
np.random.seed(42)
m = 100                                # número de observações
X = 2 * np.random.rand(m, 1)           # uma variável, em coluna
y = 4 + 3 * X + np.random.randn(m, 1)  # y = 4 + 3x + ruído

plt.figure(figsize=(6, 4))
plt.plot(X, y, "b.")
plt.xlabel("$x_1$")
plt.ylabel("$y$", rotation=0)
plt.axis([0, 2, 0, 15])
plt.grid(alpha=0.25)
plt.show()

### 1.2 Aplicar a equação normal

Antes de usar a fórmula, adicionamos a `X` uma coluna de 1s (o "$x_0 = 1$"). Assim o intercepto $\theta_0$ é tratado como um peso qualquer. O operador `@` faz multiplicação de matrizes e `np.linalg.inv` calcula a inversa.

In [ ]:
from sklearn.preprocessing import add_dummy_feature

X_b = add_dummy_feature(X)  # adiciona x0 = 1 a cada observação
theta_otimo = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y

print("theta estimado (intercepto, inclinação):", theta_otimo.ravel().round(3))
print("theta verdadeiro:                       [4, 3]")

Os valores estimados ficam **perto** de 4 e 3, mas não iguais: o ruído dos dados impede recuperar os parâmetros exatos. Com os parâmetros, fazemos previsões para dois pontos novos e desenhamos a reta:

In [ ]:
X_new = np.array([[0], [2]])
X_new_b = add_dummy_feature(X_new)
y_predict = X_new_b @ theta_otimo
print("Previsões para x = 0 e x = 2:", y_predict.ravel().round(2))

plt.figure(figsize=(6, 4))
plt.plot(X_new, y_predict, "r-", label="Previsões")
plt.plot(X, y, "b.")
plt.xlabel("$x_1$")
plt.ylabel("$y$", rotation=0)
plt.axis([0, 2, 0, 15])
plt.grid(alpha=0.25)
plt.legend(loc="upper left")
plt.show()

### 1.3 O mesmo resultado com o scikit-learn

O `LinearRegression` faz exatamente isso (e já cuida do intercepto). Na Monitoria 2 usamos essa classe sem saber o que havia dentro; agora sabemos.

In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(X, y)
print("Intercepto:", lin_reg.intercept_.round(3), "| Coeficiente:", lin_reg.coef_.round(3))
print("Previsões para x = 0 e x = 2:", lin_reg.predict(X_new).ravel().round(2))

Por baixo dos panos, o `LinearRegression` não inverte a matriz $\mathbf{X}^\top\mathbf{X}$. Ele usa a **pseudoinversa** de $\mathbf{X}$ (calculada por *decomposição em valores singulares*, SVD). Isso é mais estável numericamente e funciona mesmo quando $\mathbf{X}^\top\mathbf{X}$ não tem inversa (por exemplo, quando duas variáveis são cópias uma da outra). Você pode chamar essas funções diretamente:

In [ ]:
theta_svd, residuos, posto, valores_singulares = np.linalg.lstsq(X_b, y, rcond=1e-6)
print("lstsq:", theta_svd.ravel().round(3))
print("pinv: ", (np.linalg.pinv(X_b) @ y).ravel().round(3))

### 1.4 Aplicação: PIB e expectativa de vida

Voltemos aos dados do Banco Mundial das monitorias anteriores. Ajustamos uma regressão linear simples: expectativa de vida em função do **log do PIB per capita**. Usamos o log porque um aumento de renda faz muito mais diferença para um país pobre do que para um rico.

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/liviameinhardt/curso-decisoes-negocios-dados-epge/main/dados/indicadores_banco_mundial.csv")

paises = df[df["ano"] == 2022].dropna(subset=["expectativa_vida", "pib_per_capita_usd"]).copy()
paises["log_pib"] = np.log10(paises["pib_per_capita_usd"])

reg_paises = LinearRegression()
reg_paises.fit(paises[["log_pib"]], paises["expectativa_vida"])
print("Intercepto:", round(reg_paises.intercept_, 2))
print("Coeficiente:", round(reg_paises.coef_[0], 2))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(paises["log_pib"], paises["expectativa_vida"], alpha=0.6)
xs = np.linspace(paises["log_pib"].min(), paises["log_pib"].max(), 100)
ax.plot(xs, reg_paises.predict(pd.DataFrame({"log_pib": xs})), "r-", linewidth=2)
ax.set_xlabel("log10 do PIB per capita (US$)")
ax.set_ylabel("Expectativa de vida (anos)")
ax.grid(alpha=0.25)
plt.show()

**Como ler o coeficiente:** como a variável está em $\log_{10}$, somar 1 a `log_pib` significa **multiplicar o PIB per capita por 10**. O coeficiente é, portanto, a diferença média de expectativa de vida entre países cujo PIB per capita difere por um fator de 10. É uma associação, não um efeito causal.

**Sua vez:** estime a mesma regressão para o ano de 2000 e compare o coeficiente com o de 2022. A relação entre renda e longevidade mudou?

In [ ]:
# Escreva sua solução aqui

### 1.5 Limitação da equação normal

O custo da equação normal cresce rapidamente com o **número de variáveis** (mais que ao quadrado). Com poucas variáveis e muitas observações, funciona bem. Com milhares de variáveis, fica lenta. Precisamos de outro jeito de treinar o modelo, o que nos leva ao próximo bloco.

## 2. Gradiente descendente

O **gradiente descendente** é um algoritmo genérico de otimização que ajusta os parâmetros **aos poucos**, sempre na direção que reduz o erro. A analogia clássica: você está perdido na montanha, com neblina, e só sente a inclinação do chão sob os pés. A melhor estratégia é descer sempre pela maior inclinação.

O algoritmo:
1. Comece com valores **aleatórios** para $\theta$.
2. Calcule o **gradiente** do erro (a direção de maior subida). Para o MSE:
$$\nabla_\theta \text{MSE}(\theta) = \frac{2}{m}\mathbf{X}^\top\left(\mathbf{X}\theta - \mathbf{y}\right)$$
3. Dê um passo **na direção oposta**, de tamanho controlado pela **taxa de aprendizado** $\eta$ (*learning rate*):
$$\theta \leftarrow \theta - \eta\,\nabla_\theta \text{MSE}(\theta)$$
4. Repita até o erro parar de diminuir (convergência).

Duas propriedades importantes do MSE na regressão linear: a função de custo é **convexa** (um único "vale", sem mínimos locais enganosos) e **contínua**. Se a taxa de aprendizado não for muito alta, o gradiente descendente chega ao mínimo global.

### 2.1 Gradiente descendente em lote (*batch*)

Em cada passo, usamos **todas** as observações para calcular o gradiente. Cada passada completa pelos dados é uma **época** (*epoch*).

In [ ]:
eta = 0.1        # taxa de aprendizado
n_epocas = 1000
m = len(X_b)

np.random.seed(42)
theta = np.random.randn(2, 1)  # parâmetros iniciais aleatórios

for epoca in range(n_epocas):
    gradientes = 2 / m * X_b.T @ (X_b @ theta - y)
    theta = theta - eta * gradientes

print("Gradiente descendente:", theta.ravel().round(3))
print("Equação normal:       ", theta_otimo.ravel().round(3))

O resultado é praticamente idêntico ao da equação normal.

### 2.2 O papel da taxa de aprendizado

A taxa $\eta$ define o tamanho do passo. Veja as primeiras 20 épocas com três valores diferentes (as linhas mais escuras vêm de épocas mais avançadas):

- **$\eta$ muito pequeno** (0,02): o algoritmo converge, mas **devagar**.
- **$\eta$ adequado** (0,1): chega rápido à solução.
- **$\eta$ muito grande** (0,5): os passos "pulam" o vale e o algoritmo **diverge**.

In [ ]:
def plotar_gd(theta, eta, n_epocas=1000, n_mostrar=20):
    """Roda o gradiente descendente em lote e desenha as primeiras épocas."""
    caminho = []
    plt.plot(X, y, "b.")
    for epoca in range(n_epocas):
        if epoca < n_mostrar:
            cor = mpl.colors.rgb2hex(plt.cm.OrRd(epoca / n_mostrar + 0.15))
            plt.plot(X_new, X_new_b @ theta, linestyle="solid", color=cor)
        gradientes = 2 / m * X_b.T @ (X_b @ theta - y)
        theta = theta - eta * gradientes
        caminho.append(theta)
    plt.xlabel("$x_1$")
    plt.axis([0, 2, 0, 15])
    plt.grid(alpha=0.25)
    plt.title(fr"$\eta = {eta}$")
    return caminho

np.random.seed(42)
theta = np.random.randn(2, 1)  # mesma inicialização para os três

plt.figure(figsize=(10, 4))
plt.subplot(131)
plotar_gd(theta, eta=0.02)
plt.ylabel("$y$", rotation=0)
plt.subplot(132)
theta_path_bgd = plotar_gd(theta, eta=0.1)
plt.gca().axes.yaxis.set_ticklabels([])
plt.subplot(133)
plt.gca().axes.yaxis.set_ticklabels([])
plotar_gd(theta, eta=0.5)
plt.tight_layout()
plt.show()

**Dica prática:** para achar um bom $\eta$, use uma busca em grade (`GridSearchCV`). Para o número de épocas, escolha um valor grande e **interrompa** quando o gradiente ficar menor que um limite pequeno (*tolerância*), o que o scikit-learn faz por você.

### 2.3 Padronizar as variáveis acelera o gradiente descendente

Se as variáveis têm **escalas muito diferentes** (por exemplo, renda em milhares e idade em dezenas), a superfície do erro vira um "vale comprido e estreito" e o algoritmo zigue-zagueia devagar. Com as variáveis padronizadas (`StandardScaler`), o vale fica arredondado e o caminho é direto. A equação normal não sofre com isso, mas o gradiente descendente sim. **Padronize sempre que usar gradiente descendente.**

### 2.4 Gradiente descendente estocástico (SGD)

O gradiente em lote fica lento com muitos dados, porque usa todas as observações em cada passo. O **estocástico** usa **uma observação sorteada** por passo. É muito mais rápido por passo, e funciona com bases que nem cabem na memória. O custo é que o caminho é **irregular**: o erro sobe e desce, e o algoritmo fica "quicando" perto do mínimo sem nunca parar nele.

Essa irregularidade tem um lado bom: pode ajudar a escapar de mínimos locais em funções não convexas. Para que o SGD se estabilize, reduzimos a taxa de aprendizado ao longo do tempo (*learning schedule*), como um recozimento: passos grandes no início, pequenos no fim.

In [ ]:
theta_path_sgd = []  # guardamos o caminho de theta para o gráfico final

n_epocas = 50
t0, t1 = 5, 50  # hiperparâmetros do cronograma de aprendizado

def cronograma(t):
    return t0 / (t + t1)

np.random.seed(42)
theta = np.random.randn(2, 1)

n_mostrar = 20
plt.figure(figsize=(6, 4))

for epoca in range(n_epocas):
    for iteracao in range(m):
        if epoca == 0 and iteracao < n_mostrar:  # só para desenhar os primeiros passos
            cor = mpl.colors.rgb2hex(plt.cm.OrRd(iteracao / n_mostrar + 0.15))
            plt.plot(X_new, X_new_b @ theta, color=cor)

        i = np.random.randint(m)                 # sorteia UMA observação
        xi = X_b[i : i + 1]
        yi = y[i : i + 1]
        gradientes = 2 * xi.T @ (xi @ theta - yi)  # no SGD não dividimos por m
        eta = cronograma(epoca * m + iteracao)
        theta = theta - eta * gradientes
        theta_path_sgd.append(theta)

plt.plot(X, y, "b.")
plt.xlabel("$x_1$")
plt.ylabel("$y$", rotation=0)
plt.axis([0, 2, 0, 15])
plt.grid(alpha=0.25)
plt.show()

print("SGD:", theta.ravel().round(3))

O resultado fica perto do obtido pelo gradiente em lote e pela equação normal. O scikit-learn oferece o `SGDRegressor` (você já viu o `SGDClassifier` na Monitoria 2). Por padrão ele otimiza o MSE; aqui desligamos a regularização (`penalty=None`) para comparar com os métodos anteriores.

In [ ]:
from sklearn.linear_model import SGDRegressor

sgd_reg = SGDRegressor(max_iter=1000, tol=1e-5, penalty=None, eta0=0.01,
                       n_iter_no_change=100, random_state=42)
sgd_reg.fit(X, y.ravel())  # y.ravel() porque fit() espera o alvo em 1 dimensão

print("Intercepto:", sgd_reg.intercept_.round(3), "| Coeficiente:", sgd_reg.coef_.round(3))

### 2.5 Gradiente descendente em mini-lotes (*mini-batch*)

O **mini-batch** é o meio-termo: em cada passo, usa um pequeno grupo de observações (por exemplo, 20). É mais estável que o estocástico e mais rápido que o lote completo. É a versão mais usada em deep learning, em parte porque as placas de vídeo (GPUs) são muito boas em calcular lotes de uma vez.

In [ ]:
from math import ceil

n_epocas = 50
tamanho_lote = 20
lotes_por_epoca = ceil(m / tamanho_lote)

def cronograma_mb(t, t0=200, t1=1000):
    return t0 / (t + t1)

np.random.seed(42)
theta = np.random.randn(2, 1)

theta_path_mgd = []
for epoca in range(n_epocas):
    indices = np.random.permutation(m)       # embaralha a cada época
    X_emb, y_emb = X_b[indices], y[indices]
    for iteracao in range(lotes_por_epoca):
        i = iteracao * tamanho_lote
        xi = X_emb[i : i + tamanho_lote]
        yi = y_emb[i : i + tamanho_lote]
        gradientes = 2 / len(xi) * xi.T @ (xi @ theta - yi)
        eta = cronograma_mb(epoca * lotes_por_epoca + iteracao)
        theta = theta - eta * gradientes
        theta_path_mgd.append(theta)

print("Mini-batch:", theta.ravel().round(3))

### 2.6 Comparar os três caminhos

O gráfico mostra o caminho de $(\theta_0, \theta_1)$ em cada algoritmo, do ponto inicial até o mínimo.

In [ ]:
caminho_bgd = np.array(theta_path_bgd)
caminho_sgd = np.array(theta_path_sgd)
caminho_mgd = np.array(theta_path_mgd)

plt.figure(figsize=(7, 4))
plt.plot(caminho_sgd[:, 0], caminho_sgd[:, 1], "r-s", linewidth=1, label="Estocástico")
plt.plot(caminho_mgd[:, 0], caminho_mgd[:, 1], "g-+", linewidth=2, label="Mini-batch")
plt.plot(caminho_bgd[:, 0], caminho_bgd[:, 1], "b-o", linewidth=3, label="Lote (batch)")
plt.legend(loc="upper left")
plt.xlabel(r"$\theta_0$")
plt.ylabel(r"$\theta_1$   ", rotation=0)
plt.axis([2.6, 4.6, 2.3, 3.4])
plt.grid(alpha=0.25)
plt.show()

### Resumo: qual algoritmo usar?

| Algoritmo | Muitas observações | Muitas variáveis | Precisa de $\eta$ | Precisa padronizar |
|---|---|---|---|---|
| Equação normal / SVD | Lento (SVD) | **Lento** | Não | Não |
| Gradiente em lote | **Lento** | Rápido | Sim | Sim |
| Gradiente estocástico | Rápido | Rápido | Sim | Sim |
| Mini-batch | Rápido | Rápido | Sim | Sim |

Todos chegam a modelos muito parecidos em problemas convexos como a regressão linear. Em muitos dados, o estocástico e o mini-batch são os preferidos.

**Interprete**
- Por que o caminho do SGD é mais irregular que o do lote? Qual é o ganho em troca?
- O que acontece se você aumentar `eta` para 0,5 no gradiente em lote? E se reduzir para 0,001?

**Sua vez:** na célula do gradiente em lote da seção 2.1, mude `n_epocas` para 10 e depois para 100. Com quantas épocas o resultado já fica próximo da equação normal?

In [ ]:
# Escreva sua solução aqui

## 3. Regressão polinomial e curvas de aprendizado

E se os dados **não** seguirem uma reta? Podemos ainda usar um modelo linear, com um truque: **criar novas variáveis** que são potências das originais ($x^2$, $x^3$, ...). O modelo continua linear **nos parâmetros** e ajusta uma curva. Isso é a **regressão polinomial**.

### 3.1 Dados quadráticos

In [ ]:
np.random.seed(42)
m_quad = 100
X_quad = 6 * np.random.rand(m_quad, 1) - 3
y_quad = 0.5 * X_quad ** 2 + X_quad + 2 + np.random.randn(m_quad, 1)  # y = 0,5x² + x + 2 + ruído

plt.figure(figsize=(6, 4))
plt.plot(X_quad, y_quad, "b.")
plt.xlabel("$x_1$")
plt.ylabel("$y$", rotation=0)
plt.axis([-3, 3, 0, 10])
plt.grid(alpha=0.25)
plt.show()

Uma reta claramente não serve. Usamos o `PolynomialFeatures` para acrescentar o quadrado de cada variável e depois ajustamos a regressão linear como sempre:

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly_features = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly_features.fit_transform(X_quad)

print("Primeira observação original:  ", X_quad[0].round(3))
print("Primeira com o termo quadrático:", X_poly[0].round(3))

lin_reg = LinearRegression()
lin_reg.fit(X_poly, y_quad)
print("Intercepto:", lin_reg.intercept_.round(2), "| Coeficientes (x, x²):", lin_reg.coef_.round(2))
print("Verdadeiros: intercepto 2 | coeficientes (1, 0,5)")

In [ ]:
X_curva = np.linspace(-3, 3, 100).reshape(100, 1)
y_curva = lin_reg.predict(poly_features.transform(X_curva))

plt.figure(figsize=(6, 4))
plt.plot(X_quad, y_quad, "b.")
plt.plot(X_curva, y_curva, "r-", linewidth=2, label="Previsões")
plt.xlabel("$x_1$")
plt.ylabel("$y$", rotation=0)
plt.legend(loc="upper left")
plt.axis([-3, 3, 0, 10])
plt.grid(alpha=0.25)
plt.show()

**Com várias variáveis**, o `PolynomialFeatures(degree=d)` cria também as **interações** entre elas. Com $n$ variáveis e grau $d$, o número de colunas explode (é a combinação de $n + d$ tomados $d$ a $d$). Cuidado com o grau!

### 3.2 Grau alto demais: *underfitting* e *overfitting*

Vamos comparar três modelos nos mesmos dados: grau 1 (reta), grau 2 (o certo) e grau 300 (absurdo).

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

plt.figure(figsize=(6, 4))

for estilo, largura, grau in (("r-+", 2, 1), ("b--", 2, 2), ("g-", 1, 300)):
    modelo_poly = make_pipeline(
        PolynomialFeatures(degree=grau, include_bias=False),
        StandardScaler(),
        LinearRegression(),
    )
    modelo_poly.fit(X_quad, y_quad)
    plt.plot(X_curva, modelo_poly.predict(X_curva), estilo,
             label=f"grau {grau}", linewidth=largura)

plt.plot(X_quad, y_quad, "b.", linewidth=3)
plt.legend(loc="upper left")
plt.xlabel("$x_1$")
plt.ylabel("$y$", rotation=0)
plt.axis([-3, 3, 0, 10])
plt.grid(alpha=0.25)
plt.show()

- **Grau 1 (reta): *underfitting*** (subajuste). O modelo é simples demais e erra até no treino.
- **Grau 2: bom ajuste.** Captura o padrão sem decorar o ruído.
- **Grau 300: *overfitting*** (sobreajuste). A curva passa por cima dos pontos do treino e faz previsões absurdas entre eles.

Em negócios, o overfitting é o risco silencioso: o modelo parece ótimo na base histórica e falha com clientes novos. Já vimos isso na Monitoria 2, com a árvore de decisão que tinha erro quase zero no treino.

### 3.3 Como saber se o modelo está sub ou sobreajustado?

Duas formas:

1. **Validação cruzada** (Monitoria 2): bom desempenho no treino e ruim na validação → overfitting. Ruim nos dois → underfitting.
2. **Curvas de aprendizado:** gráfico do erro de treino e de validação **em função do tamanho da base de treino**. O `learning_curve` do scikit-learn treina o modelo em subconjuntos crescentes e mede o erro com validação cruzada.

Comecemos pela reta (que sabemos ser um modelo simples demais):

In [ ]:
from sklearn.model_selection import learning_curve

tamanhos, scores_treino, scores_valid = learning_curve(
    LinearRegression(), X_quad, y_quad, train_sizes=np.linspace(0.01, 1.0, 40),
    cv=5, scoring="neg_root_mean_squared_error")
erros_treino = -scores_treino.mean(axis=1)
erros_valid = -scores_valid.mean(axis=1)

plt.figure(figsize=(6, 4))
plt.plot(tamanhos, erros_treino, "r-+", linewidth=2, label="treino")
plt.plot(tamanhos, erros_valid, "b-", linewidth=3, label="validação")
plt.xlabel("Tamanho da base de treino")
plt.ylabel("RMSE")
plt.grid(alpha=0.25)
plt.legend(loc="upper right")
plt.axis([0, 80, 0, 2.5])
plt.show()

**Padrão de *underfitting*:** as duas curvas **se aproximam e estabilizam num nível alto**. O erro de treino começa em zero (com poucos pontos, uma reta passa por todos) e sobe; o de validação desce. Como o modelo é simples demais, **mais dados não ajudam**. A saída é um modelo mais flexível (mais variáveis, grau maior) ou menos regularização.

Agora um polinômio de grau 10 (flexível demais para estes dados):

In [ ]:
poly10 = make_pipeline(
    PolynomialFeatures(degree=10, include_bias=False),
    LinearRegression())

tamanhos, scores_treino, scores_valid = learning_curve(
    poly10, X_quad, y_quad, train_sizes=np.linspace(0.01, 1.0, 40),
    cv=5, scoring="neg_root_mean_squared_error")
erros_treino = -scores_treino.mean(axis=1)
erros_valid = -scores_valid.mean(axis=1)

plt.figure(figsize=(6, 4))
plt.plot(tamanhos, erros_treino, "r-+", linewidth=2, label="treino")
plt.plot(tamanhos, erros_valid, "b-", linewidth=3, label="validação")
plt.legend(loc="upper right")
plt.xlabel("Tamanho da base de treino")
plt.ylabel("RMSE")
plt.grid(alpha=0.25)
plt.axis([0, 80, 0, 2.5])
plt.show()

**Padrão de *overfitting*:** o erro de treino é **muito menor** que o de validação, com uma **lacuna** entre as curvas. Aqui, **mais dados ajudam**: a lacuna diminui com a base maior. Outras saídas: simplificar o modelo ou **regularizar** (próximo bloco).

### O dilema viés–variância

O erro de generalização de um modelo se decompõe em três partes:

- **Viés** (*bias*): erro por suposições erradas, como achar que é linear quando é quadrático. Viés alto → *underfitting*.
- **Variância:** sensibilidade do modelo a pequenas mudanças na base de treino. Modelos muito flexíveis têm variância alta → *overfitting*.
- **Erro irredutível:** o ruído dos próprios dados. Nenhum modelo elimina.

Aumentar a complexidade **reduz o viés e aumenta a variância**. O trabalho de quem modela é encontrar o ponto de equilíbrio.

**Sua vez:** refaça o gráfico de grau 300 com `degree=3`, `4` e `5`. Em qual grau a curva "estraga"? Isso acontece de repente ou aos poucos?

In [ ]:
# Escreva sua solução aqui

---
## ☕ Intervalo (10 min)

---
## 4. Modelos regularizados

**Regularizar** é restringir o modelo para reduzir o overfitting. Nos modelos lineares, isso é feito **limitando o tamanho dos pesos**: acrescentamos à função de custo uma penalidade que cresce com os pesos. O hiperparâmetro $\alpha$ controla a força da penalidade (com $\alpha = 0$ voltamos à regressão linear comum).

Regra de ouro: **padronize as variáveis antes de regularizar.** A penalidade trata todos os pesos igualmente; se uma variável estiver em escala pequena, seu peso precisa ser grande e será injustamente punido.

Vamos usar uma base pequena e ruidosa, em que o overfitting é fácil de provocar:

In [ ]:
np.random.seed(42)
m_pq = 20
X_pq = 3 * np.random.rand(m_pq, 1)
y_pq = 1 + 0.5 * X_pq + np.random.randn(m_pq, 1) / 1.5
X_pq_novo = np.linspace(0, 3, 100).reshape(100, 1)

plt.figure(figsize=(6, 4))
plt.plot(X_pq, y_pq, ".")
plt.xlabel("$x_1$")
plt.ylabel("$y$  ", rotation=0)
plt.axis([0, 3, 0, 3.5])
plt.grid(alpha=0.25)
plt.show()

### 4.1 Regressão Ridge (penalidade $\ell_2$)

A **Ridge** soma ao MSE a soma dos **quadrados** dos pesos (sem contar o intercepto):

$$J(\theta) = \text{MSE}(\theta) + \alpha\,\frac{1}{2}\sum_{j=1}^{n}\theta_j^2$$

O efeito é **encolher** os pesos em direção a zero, sem zerá-los. Quanto maior o $\alpha$, mais "reta" fica a previsão.

In [ ]:
from sklearn.linear_model import Ridge

ridge_reg = Ridge(alpha=0.1, solver="cholesky")
ridge_reg.fit(X_pq, y_pq)
print("Previsão para x = 1,5:", ridge_reg.predict([[1.5]]).round(3))

A função abaixo ajusta um modelo regularizado para vários valores de $\alpha$ e desenha as curvas. À esquerda, modelo linear; à direita, polinômio de grau 10 (propenso a overfitting). Ela serve também para a Lasso, mais adiante.

In [ ]:
def plotar_modelo(classe_modelo, polinomial, alphas, **kwargs_modelo):
    plt.plot(X_pq, y_pq, "b.", linewidth=3)
    for alpha, estilo in zip(alphas, ("b:", "g--", "r-")):
        if alpha > 0:
            modelo = classe_modelo(alpha, **kwargs_modelo)
        else:
            modelo = LinearRegression()
        if polinomial:
            modelo = make_pipeline(
                PolynomialFeatures(degree=10, include_bias=False),
                StandardScaler(),
                modelo)
        modelo.fit(X_pq, y_pq)
        plt.plot(X_pq_novo, modelo.predict(X_pq_novo), estilo, linewidth=2,
                 label=fr"$\alpha = {alpha}$")
    plt.legend(loc="upper left")
    plt.xlabel("$x_1$")
    plt.axis([0, 3, 0, 3.5])
    plt.grid(alpha=0.25)

plt.figure(figsize=(9, 3.5))
plt.subplot(121)
plotar_modelo(Ridge, polinomial=False, alphas=(0, 10, 100), random_state=42)
plt.ylabel("$y$  ", rotation=0)
plt.subplot(122)
plotar_modelo(Ridge, polinomial=True, alphas=(0, 1e-5, 1), random_state=42)
plt.gca().axes.yaxis.set_ticklabels([])
plt.tight_layout()
plt.show()

**Interprete**
- À esquerda: o que acontece com a inclinação da reta quando $\alpha$ cresce?
- À direita: o polinômio de grau 10 sem regularização ($\alpha = 0$) oscila muito. Com $\alpha = 1$ a curva fica suave. A regularização "domou" o overfitting?
- Se $\alpha$ for grande demais, o modelo vira uma reta horizontal. Isso é *underfitting*.

Como a Ridge é um modelo linear, também pode ser treinada por gradiente descendente. No `SGDRegressor`, `penalty="l2"` faz isso:

In [ ]:
sgd_ridge = SGDRegressor(penalty="l2", alpha=0.1 / m_pq, tol=None,
                         max_iter=1000, eta0=0.01, random_state=42)
sgd_ridge.fit(X_pq, y_pq.ravel())
print("Ridge por SGD:", sgd_ridge.predict([[1.5]]).round(3))

### 4.2 Regressão Lasso (penalidade $\ell_1$)

A **Lasso** (*least absolute shrinkage and selection operator*) usa a soma dos **valores absolutos** dos pesos:

$$J(\theta) = \text{MSE}(\theta) + \alpha\sum_{j=1}^{n}|\theta_j|$$

A diferença prática em relação à Ridge: a Lasso tende a **zerar completamente** os pesos das variáveis menos importantes. O resultado é um modelo **esparso**, que faz **seleção automática de variáveis**. Isso é útil quando há muitas variáveis candidatas e poucas realmente importam.

In [ ]:
from sklearn.linear_model import Lasso

lasso_reg = Lasso(alpha=0.1)
lasso_reg.fit(X_pq, y_pq)
print("Lasso — previsão para x = 1,5:", lasso_reg.predict([[1.5]]).round(3))

plt.figure(figsize=(9, 3.5))
plt.subplot(121)
plotar_modelo(Lasso, polinomial=False, alphas=(0, 0.1, 1), random_state=42)
plt.ylabel("$y$  ", rotation=0)
plt.subplot(122)
plotar_modelo(Lasso, polinomial=True, alphas=(0, 1e-2, 1), random_state=42)
plt.gca().axes.yaxis.set_ticklabels([])
plt.tight_layout()
plt.show()

Para ver a esparsidade na prática, ajustamos Ridge e Lasso com polinômio de grau 10 e contamos quantos coeficientes foram a **exatamente zero**:

In [ ]:
prepara_poly = make_pipeline(PolynomialFeatures(degree=10, include_bias=False), StandardScaler())
X_pq_poly = prepara_poly.fit_transform(X_pq)

ridge_poly = Ridge(alpha=1).fit(X_pq_poly, y_pq)
lasso_poly = Lasso(alpha=0.1, max_iter=100_000).fit(X_pq_poly, y_pq)

print("Coeficientes iguais a zero (de 10):")
print("  Ridge:", int((ridge_poly.coef_ == 0).sum()))
print("  Lasso:", int((lasso_poly.coef_ == 0).sum()))

### 4.3 Elastic Net

A **Elastic Net** é uma combinação das duas penalidades, controlada por `l1_ratio` ($r$): com $r = 0$ equivale à Ridge e com $r = 1$, à Lasso.

$$J(\theta) = \text{MSE}(\theta) + r\,\alpha\sum_{j}|\theta_j| + \frac{1-r}{2}\,\alpha\sum_{j}\theta_j^2$$

In [ ]:
from sklearn.linear_model import ElasticNet

elastic_net = ElasticNet(alpha=0.1, l1_ratio=0.5)
elastic_net.fit(X_pq, y_pq)
print("Elastic Net — previsão para x = 1,5:", elastic_net.predict([[1.5]]).round(3))

### Qual usar?

| Modelo | Quando usar |
|---|---|
| Regressão linear sem penalidade | Raramente. Pelo menos um pouco de regularização quase sempre ajuda. |
| **Ridge** | Boa opção padrão. Todas as variáveis podem importar um pouco. |
| **Lasso** | Suspeita de que poucas variáveis importam e você quer seleção automática. |
| **Elastic Net** | Muitas variáveis, ou variáveis fortemente correlacionadas entre si (quando a Lasso se comporta de forma instável). |

O $\alpha$ é um hiperparâmetro e se escolhe com validação cruzada, como fizemos no `GridSearchCV` da Monitoria 2. Existem as versões `RidgeCV`, `LassoCV` e `ElasticNetCV`, que já fazem essa busca.

**Sua vez:** volte aos dados do Banco Mundial (seção 1.4). Crie uma regressão polinomial de grau 6 de `expectativa_vida` em `log_pib` com `Ridge(alpha=1)` e compare, por validação cruzada (`cross_val_score` com RMSE), com a mesma regressão sem regularização.

In [ ]:
# Dica:
# from sklearn.model_selection import cross_val_score
# X_pib = paises[["log_pib"]]; y_vida = paises["expectativa_vida"]
# modelo = make_pipeline(PolynomialFeatures(degree=6, include_bias=False), StandardScaler(), Ridge(alpha=1))
# -cross_val_score(modelo, X_pib, y_vida, cv=5, scoring="neg_root_mean_squared_error").mean()

### 4.4 Parada antecipada (*early stopping*)

Outra forma de regularizar algoritmos iterativos, como o gradiente descendente: **parar o treino quando o erro de validação para de cair**. Com o passar das épocas, o erro de treino só diminui, mas o de validação desce até um mínimo e depois **volta a subir** (o modelo começa a decorar o treino). Guardamos o melhor modelo e paramos ali.

Para exagerar o efeito, usamos o mesmo problema quadrático com um polinômio de grau 90, que tem tudo para sofrer overfitting. Usamos `partial_fit`, que faz **uma época** de treino por chamada.

In [ ]:
from copy import deepcopy
from sklearn.metrics import root_mean_squared_error

X_train, y_train = X_quad[: m_quad // 2], y_quad[: m_quad // 2, 0]
X_valid, y_valid = X_quad[m_quad // 2 :], y_quad[m_quad // 2 :, 0]

preprocessamento = make_pipeline(PolynomialFeatures(degree=90, include_bias=False),
                                 StandardScaler())
X_train_prep = preprocessamento.fit_transform(X_train)
X_valid_prep = preprocessamento.transform(X_valid)

sgd_reg = SGDRegressor(penalty=None, eta0=0.002, random_state=42)
n_epocas = 500
melhor_rmse_valid = float("inf")
erros_treino, erros_valid = [], []

for epoca in range(n_epocas):
    sgd_reg.partial_fit(X_train_prep, y_train)  # uma época de treino
    rmse_valid = root_mean_squared_error(y_valid, sgd_reg.predict(X_valid_prep))
    rmse_treino = root_mean_squared_error(y_train, sgd_reg.predict(X_train_prep))
    erros_valid.append(rmse_valid)
    erros_treino.append(rmse_treino)
    if rmse_valid < melhor_rmse_valid:          # guarda o melhor modelo até agora
        melhor_rmse_valid = rmse_valid
        melhor_modelo = deepcopy(sgd_reg)

melhor_epoca = int(np.argmin(erros_valid))

plt.figure(figsize=(7, 4))
plt.plot(erros_valid, "b-", linewidth=3, label="Validação")
plt.plot(erros_treino, "r--", linewidth=2, label="Treino")
plt.plot([0, n_epocas], [melhor_rmse_valid, melhor_rmse_valid], "k:", linewidth=2)
plt.plot(melhor_epoca, melhor_rmse_valid, "bo")
plt.annotate("Melhor modelo", xy=(melhor_epoca, melhor_rmse_valid),
             xytext=(melhor_epoca, melhor_rmse_valid + 0.5), ha="center",
             arrowprops=dict(facecolor="black", shrink=0.05))
plt.legend(loc="upper right")
plt.xlabel("Época")
plt.ylabel("RMSE")
plt.axis([0, n_epocas, 0, 3.5])
plt.grid(alpha=0.25)
plt.show()

print(f"Melhor época: {melhor_epoca} | RMSE de validação: {melhor_rmse_valid:.3f}")

**Interprete**
- Em que época o erro de validação é mínimo? O que acontece com os erros de treino e de validação depois dela?
- O erro de validação oscila e pode subir por um tempo antes de cair de novo. Por que **não** devemos parar na primeira vez que ele sobe? (Por isso guardamos o melhor modelo e continuamos treinando.)
- Lembre: a validação aqui participou da escolha da época. Para medir o desempenho final, ainda precisaríamos de um **teste** separado.

Geralmente, o *early stopping* é simples e eficiente, e Géron o chama de "um almoço grátis".

## 5. Regressão logística

A **regressão logística** é usada para **classificação**, apesar do nome. Ela estima a **probabilidade** de uma observação pertencer à classe positiva. Se a probabilidade passar de 50%, prevê "positivo"; caso contrário, "negativo". Na Monitoria 2 usamos classificadores que davam uma pontuação e um *threshold*. A regressão logística é o caso em que essa pontuação vira uma probabilidade de verdade.

### 5.1 Como estima probabilidades

Como na regressão linear, calcula-se a soma ponderada das variáveis, $t = \theta^\top\mathbf{x}$ (a "pontuação" ou *logit*). Depois, passa-se por uma função **logística** (sigmoide), que comprime qualquer número real no intervalo de 0 a 1:

$$\hat{p} = \sigma(t) = \frac{1}{1 + e^{-t}}$$

- $t = 0$ → $\hat{p} = 0{,}5$
- $t$ grande e positivo → $\hat{p}$ perto de 1
- $t$ grande e negativo → $\hat{p}$ perto de 0

In [ ]:
lim = 6
t = np.linspace(-lim, lim, 100)
sig = 1 / (1 + np.exp(-t))

plt.figure(figsize=(8, 3))
plt.plot([-lim, lim], [0, 0], "k-")
plt.plot([-lim, lim], [0.5, 0.5], "k:")
plt.plot([-lim, lim], [1, 1], "k:")
plt.plot([0, 0], [-1.1, 1.1], "k-")
plt.plot(t, sig, "b-", linewidth=2, label=r"$\sigma(t) = \dfrac{1}{1 + e^{-t}}$")
plt.xlabel("t")
plt.legend(loc="upper left")
plt.axis([-lim, lim, -0.1, 1.1])
plt.gca().set_yticks([0, 0.25, 0.5, 0.75, 1])
plt.grid(alpha=0.25)
plt.show()

### 5.2 Como é treinada

A função de custo é a **entropia cruzada** (*log loss*), que pune fortemente quando o modelo tem **certeza e erra**:

$$J(\theta) = -\frac{1}{m}\sum_{i=1}^{m}\left[y^{(i)}\log\hat{p}^{(i)} + \left(1-y^{(i)}\right)\log\left(1-\hat{p}^{(i)}\right)\right]$$

Não existe equação normal para ela, mas a função é **convexa**, então o gradiente descendente chega ao mínimo global. O scikit-learn usa solucionadores mais sofisticados (por padrão, `lbfgs`) e, como na Ridge, aplica **regularização $\ell_2$ por padrão**. O hiperparâmetro é o **`C`**, o inverso de $\alpha$: **quanto maior o `C`, menor a regularização**.

### 5.3 Fronteira de decisão com o conjunto Iris

Usamos o clássico conjunto **Iris**: 150 flores de 3 espécies, com medidas de pétalas e sépalas. Primeiro, vamos detectar uma espécie (*Iris virginica*) usando só a **largura da pétala**.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
display(iris.data.head(3))
print("Espécies:", iris.target_names.tolist())
print("Alvo (0, 1 ou 2) das primeiras linhas:", iris.target.head(3).tolist())

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X_iris = iris.data[["petal width (cm)"]].values
y_iris = iris.target_names[iris.target] == "virginica"  # True se for virginica
X_train, X_test, y_train, y_test = train_test_split(X_iris, y_iris, random_state=42)

log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train, y_train)
print("Acurácia no teste:", round(log_reg.score(X_test, y_test), 3))

In [ ]:
X_nova = np.linspace(0, 3, 1000).reshape(-1, 1)
y_proba = log_reg.predict_proba(X_nova)
fronteira = X_nova[y_proba[:, 1] >= 0.5][0, 0]

plt.figure(figsize=(8, 3.5))
plt.plot(X_nova, y_proba[:, 0], "b--", linewidth=2, label="Prob. de NÃO ser virginica")
plt.plot(X_nova, y_proba[:, 1], "g-", linewidth=2, label="Prob. de ser virginica")
plt.plot([fronteira, fronteira], [0, 1], "k:", linewidth=2, label="Fronteira de decisão")
plt.arrow(x=fronteira, y=0.08, dx=-0.3, dy=0, head_width=0.05, head_length=0.1, fc="b", ec="b")
plt.arrow(x=fronteira, y=0.92, dx=0.3, dy=0, head_width=0.05, head_length=0.1, fc="g", ec="g")
plt.plot(X_train[y_train == 0], y_train[y_train == 0], "bs")
plt.plot(X_train[y_train == 1], y_train[y_train == 1], "g^")
plt.xlabel("Largura da pétala (cm)")
plt.ylabel("Probabilidade")
plt.legend(loc="center left", fontsize=10)
plt.axis([0, 3, -0.02, 1.02])
plt.grid(alpha=0.25)
plt.show()

print(f"Fronteira de decisão (p = 50%): largura de pétala ≈ {fronteira:.2f} cm")

Acima de ~1,65 cm de largura a flor é classificada como *virginica*; abaixo, não. Perto da fronteira o modelo tem pouca certeza. Veja as probabilidades para duas flores, uma de cada lado:

In [ ]:
print("Classes previstas para 1,7 cm e 1,5 cm:", log_reg.predict([[1.7], [1.5]]))
print("Probabilidades [não, sim]:")
print(log_reg.predict_proba([[1.7], [1.5]]).round(3))

**Conexão com a Monitoria 2:** o limite de 50% é só o padrão. Se o erro mais caro for **deixar passar** uma *virginica* (falso negativo), podemos usar um limite menor e aceitar mais falsos positivos. Em negócios, o limite certo vem do **custo dos erros**.

**Sua vez:** encontre o maior limite de probabilidade que ainda dá **recall de pelo menos 95%** no conjunto de treino. Qual a precisão com esse limite?

In [ ]:
# Dica:
# from sklearn.metrics import precision_recall_curve
# proba_treino = log_reg.predict_proba(X_train)[:, 1]
# precisoes, recalls, limites = precision_recall_curve(y_train, proba_treino)

### 5.4 Duas variáveis e fronteira linear

Com duas variáveis (comprimento e largura da pétala), a fronteira de decisão deixa de ser um ponto e vira uma **reta**. Por isso a regressão logística é um **classificador linear**. As curvas de nível mostram a probabilidade estimada.

In [ ]:
X_iris2 = iris.data[["petal length (cm)", "petal width (cm)"]].values
X_train, X_test, y_train, y_test = train_test_split(X_iris2, y_iris, random_state=42)

log_reg2 = LogisticRegression(C=2, random_state=42)
log_reg2.fit(X_train, y_train)

# grade de pontos para desenhar as probabilidades
x0, x1 = np.meshgrid(np.linspace(2.9, 7, 500).reshape(-1, 1),
                     np.linspace(0.8, 2.7, 200).reshape(-1, 1))
X_grade = np.c_[x0.ravel(), x1.ravel()]
zz = log_reg2.predict_proba(X_grade)[:, 1].reshape(x0.shape)

# a fronteira é a reta em que theta0 + theta1*x1 + theta2*x2 = 0
esq_dir = np.array([2.9, 7])
fronteira2 = -((log_reg2.coef_[0, 0] * esq_dir + log_reg2.intercept_[0]) / log_reg2.coef_[0, 1])

plt.figure(figsize=(10, 4))
plt.plot(X_train[y_train == 0, 0], X_train[y_train == 0, 1], "bs")
plt.plot(X_train[y_train == 1, 0], X_train[y_train == 1, 1], "g^")
contorno = plt.contour(x0, x1, zz, cmap=plt.cm.brg)
plt.clabel(contorno, inline=1)
plt.plot(esq_dir, fronteira2, "k--", linewidth=3)
plt.text(3.5, 1.27, "Não é virginica", color="b", ha="center")
plt.text(6.5, 2.3, "Virginica", color="g", ha="center")
plt.xlabel("Comprimento da pétala (cm)")
plt.ylabel("Largura da pétala (cm)")
plt.axis([2.9, 7, 0.8, 2.7])
plt.grid(alpha=0.25)
plt.show()

### 5.5 Regressão softmax (várias classes)

Para mais de duas classes, a regressão logística se generaliza para a **regressão softmax** (ou *logística multinomial*), **sem precisar de vários classificadores binários** (lembre do OvR e OvO da Monitoria 2). Para cada classe $k$, calcula-se uma pontuação $s_k(\mathbf{x}) = \theta_k^\top\mathbf{x}$; a função **softmax** converte as pontuações em probabilidades que **somam 1**:

$$\hat{p}_k = \frac{\exp\left(s_k(\mathbf{x})\right)}{\sum_{j=1}^{K}\exp\left(s_j(\mathbf{x})\right)}$$

O modelo prevê a classe de maior probabilidade. A função de custo é a entropia cruzada generalizada. No scikit-learn, a `LogisticRegression` já usa softmax quando o alvo tem mais de duas classes.

In [ ]:
y_iris3 = iris["target"]  # 0 = setosa, 1 = versicolor, 2 = virginica
X_train, X_test, y_train, y_test = train_test_split(X_iris2, y_iris3, random_state=42)

softmax_reg = LogisticRegression(C=30, random_state=42)
softmax_reg.fit(X_train, y_train)

print("Classe prevista para pétala 5 × 2 cm:", iris.target_names[softmax_reg.predict([[5, 2]])])
print("Probabilidades [setosa, versicolor, virginica]:", softmax_reg.predict_proba([[5, 2]]).round(2))
print("Acurácia no teste:", round(softmax_reg.score(X_test, y_test), 3))

In [ ]:
from matplotlib.colors import ListedColormap

cores_fundo = ListedColormap(["#fafab0", "#9898ff", "#a0faa0"])

x0, x1 = np.meshgrid(np.linspace(0, 8, 500).reshape(-1, 1),
                     np.linspace(0, 3.5, 200).reshape(-1, 1))
X_grade = np.c_[x0.ravel(), x1.ravel()]
proba_grade = softmax_reg.predict_proba(X_grade)
zz1 = proba_grade[:, 1].reshape(x0.shape)                  # prob. de versicolor
zz = softmax_reg.predict(X_grade).reshape(x0.shape)        # classe prevista

plt.figure(figsize=(10, 4))
plt.plot(X_iris2[y_iris3 == 2, 0], X_iris2[y_iris3 == 2, 1], "g^", label="Iris virginica")
plt.plot(X_iris2[y_iris3 == 1, 0], X_iris2[y_iris3 == 1, 1], "bs", label="Iris versicolor")
plt.plot(X_iris2[y_iris3 == 0, 0], X_iris2[y_iris3 == 0, 1], "yo", label="Iris setosa")
plt.contourf(x0, x1, zz, cmap=cores_fundo)
contorno = plt.contour(x0, x1, zz1, cmap="hot")
plt.clabel(contorno, inline=1)
plt.xlabel("Comprimento da pétala (cm)")
plt.ylabel("Largura da pétala (cm)")
plt.legend(loc="center left", fontsize=10)
plt.axis([0.5, 7, 0, 3.5])
plt.grid(alpha=0.25)
plt.show()

As regiões coloridas mostram a classe prevista; as curvas, a probabilidade de *versicolor*. As fronteiras entre classes continuam sendo **retas**.

**Interprete**
- Em que região do gráfico o modelo tem menos certeza?
- Quando usar regressão logística em vez de uma random forest? (Pense em interpretabilidade, velocidade e quantidade de dados.)

## 6. Softmax "do zero" com NumPy (*opcional*)

Este é o exercício 12 do capítulo: implementar **gradiente descendente em lote com *early stopping* para a regressão softmax**, só com NumPy. Fazer à mão ajuda a entender o que o `LogisticRegression` esconde. As equações são:

**Custo (entropia cruzada):**
$$J(\Theta) = -\frac{1}{m}\sum_{i=1}^{m}\sum_{k=1}^{K}y_k^{(i)}\log\hat{p}_k^{(i)}$$

**Gradiente:**
$$\nabla_{\theta^{(k)}}J(\Theta) = \frac{1}{m}\sum_{i=1}^{m}\left(\hat{p}_k^{(i)} - y_k^{(i)}\right)\mathbf{x}^{(i)}$$

**Dica de depuração:** escreva o **formato** (`shape`) de cada termo. A maioria dos erros em implementações assim é trocar a ordem das matrizes.

### 6.1 Dados, bias e divisão treino/validação/teste

Separamos manualmente 60% para treino, 20% para validação e 20% para teste.

In [ ]:
X_s = iris.data[["petal length (cm)", "petal width (cm)"]].values
y_s = iris["target"].values

X_com_bias = np.c_[np.ones(len(X_s)), X_s]   # coluna de 1s para o intercepto

total = len(X_com_bias)
tam_teste = int(total * 0.2)
tam_valid = int(total * 0.2)
tam_treino = total - tam_teste - tam_valid

np.random.seed(42)
indices = np.random.permutation(total)

X_tr, y_tr = X_com_bias[indices[:tam_treino]], y_s[indices[:tam_treino]]
X_va, y_va = X_com_bias[indices[tam_treino:-tam_teste]], y_s[indices[tam_treino:-tam_teste]]
X_te, y_te = X_com_bias[indices[-tam_teste:]], y_s[indices[-tam_teste:]]

print("Treino:", X_tr.shape, "| Validação:", X_va.shape, "| Teste:", X_te.shape)

### 6.2 Alvo em *one-hot* e padronização

A softmax precisa de um **vetor de probabilidades-alvo**: 1 para a classe correta e 0 para as demais (*one-hot*). E padronizamos as variáveis (menos o bias), **calculando média e desvio só no treino**, o mesmo cuidado contra vazamento de dados da Monitoria 2.

In [ ]:
def para_one_hot(y):
    return np.diag(np.ones(y.max() + 1))[y]

print("Classes:", y_tr[:5])
print(para_one_hot(y_tr[:5]))

Y_tr, Y_va, Y_te = para_one_hot(y_tr), para_one_hot(y_va), para_one_hot(y_te)

media = X_tr[:, 1:].mean(axis=0)
desvio = X_tr[:, 1:].std(axis=0)
for base in (X_tr, X_va, X_te):
    base[:, 1:] = (base[:, 1:] - media) / desvio

### 6.3 A função softmax e o treino

In [ ]:
def softmax(logits):
    exps = np.exp(logits)
    return exps / exps.sum(axis=1, keepdims=True)

n_entradas = X_tr.shape[1]           # 3 (duas variáveis + bias)
n_saidas = len(np.unique(y_tr))      # 3 classes

eta = 0.5
n_epocas = 5001
m_tr = len(X_tr)
epsilon = 1e-5                       # evita log(0)

np.random.seed(42)
Theta = np.random.randn(n_entradas, n_saidas)

for epoca in range(n_epocas):
    Y_proba = softmax(X_tr @ Theta)
    if epoca % 1000 == 0:
        perda_valid = -(Y_va * np.log(softmax(X_va @ Theta) + epsilon)).sum(axis=1).mean()
        print(f"época {epoca}: perda de validação = {perda_valid:.4f}")
    gradientes = 1 / m_tr * X_tr.T @ (Y_proba - Y_tr)
    Theta = Theta - eta * gradientes

acuracia_valid = (softmax(X_va @ Theta).argmax(axis=1) == y_va).mean()
print("Acurácia na validação:", round(acuracia_valid, 3))

### 6.4 Regularização $\ell_2$ e *early stopping*

Acrescentamos a penalidade $\ell_2$ (sem penalizar o bias, a primeira linha de `Theta`) e interrompemos o treino quando a perda de validação **começa a subir**.

In [ ]:
eta = 0.5
n_epocas = 50_001
C = 100                              # hiperparâmetro de regularização (como no sklearn)
melhor_perda = np.inf

np.random.seed(42)
Theta = np.random.randn(n_entradas, n_saidas)

for epoca in range(n_epocas):
    Y_proba = softmax(X_tr @ Theta)
    perda_xent = -(Y_va * np.log(softmax(X_va @ Theta) + epsilon)).sum(axis=1).mean()
    perda_l2 = 1 / 2 * (Theta[1:] ** 2).sum()
    perda_total = perda_xent + 1 / C * perda_l2

    if epoca % 1000 == 0:
        print(f"época {epoca}: perda = {perda_total:.4f}")
    if perda_total < melhor_perda:
        melhor_perda = perda_total
    else:
        print(f"Early stopping na época {epoca}: a perda de validação parou de cair (melhor: {melhor_perda:.4f})")
        break

    gradientes = 1 / m_tr * X_tr.T @ (Y_proba - Y_tr)
    gradientes += np.r_[np.zeros([1, n_saidas]), 1 / C * Theta[1:]]
    Theta = Theta - eta * gradientes

print("Acurácia na validação:", round((softmax(X_va @ Theta).argmax(axis=1) == y_va).mean(), 3))

Só agora, com o modelo escolhido, olhamos o **teste** (uma única vez):

In [ ]:
acuracia_teste = (softmax(X_te @ Theta).argmax(axis=1) == y_te).mean()
print("Acurácia no teste:", round(acuracia_teste, 3))

O Iris tem só 150 flores, então 30 no teste: cada erro vale mais de 3 pontos percentuais. Mude a semente (`np.random.seed`) na divisão da seção 6.1 e rode de novo: a acurácia vai variar bastante. **Com poucos dados, qualquer estimativa de desempenho é incerta.**

**Sua vez:** variando `eta` (0,01, 0,5, 5), a perda de validação ainda cai de forma estável? Associe o comportamento ao que vimos na seção 2.2.

## 7. Usar IA nesta aula

**Para aprender** — peça uma explicação com analogia de negócios:
> Explique a diferença entre regressão Ridge e Lasso para alguém sem formação em estatística. Use um exemplo de previsão de vendas com 50 variáveis, diga quando cada uma é melhor e o que significa o hiperparâmetro alpha.

**Para decidir** — descreva o problema e peça um diagnóstico:
> Treinei um modelo de regressão para prever o preço de imóveis. O RMSE no treino é R$ 20 mil e na validação cruzada é R$ 95 mil. O que isso indica? Liste três ações para melhorar, em ordem de custo, e diga como eu verificaria se funcionaram usando `learning_curve` do scikit-learn.

**Para depurar** — o código abaixo foi "gerado por IA" e dá erro. Execute, leia a mensagem e peça ajuda à IA com o prompt da Monitoria 1 (erro completo + célula + o que você esperava).

In [ ]:
# Código com erro: execute e descubra por quê
modelo_erro = LinearRegression()
modelo_erro.fit(X_quad.ravel(), y_quad)

## 8. Fechamento

Ao terminar, você deve conseguir:
- Explicar o que é "treinar" um modelo: minimizar uma função de custo.
- Usar a equação normal e saber quando ela deixa de ser prática.
- Descrever o gradiente descendente (lote, estocástico e mini-batch), o papel da taxa de aprendizado e por que padronizar as variáveis.
- Diagnosticar *underfitting* e *overfitting* com validação cruzada e curvas de aprendizado.
- Explicar o dilema viés–variância.
- Escolher entre Ridge, Lasso e Elastic Net e ajustar o hiperparâmetro de regularização.
- Usar *early stopping*.
- Interpretar uma regressão logística como estimadora de probabilidades e a softmax como sua extensão para várias classes.

### Teste seus conhecimentos

Tente responder antes de abrir cada resposta.

1. Você tem uma base com milhões de variáveis. Quais algoritmos de treino pode usar para a regressão linear?
<details><summary>Resposta</summary>
Gradiente descendente estocástico ou mini-batch (e o em lote, se a base couber na memória). A equação normal e a SVD ficam inviáveis, porque o custo cresce rapidamente com o número de variáveis.
</details>

2. Suas variáveis têm escalas muito diferentes e o gradiente descendente está lento. O que fazer?
<details><summary>Resposta</summary>
Padronizar as variáveis (<code>StandardScaler</code>). Sem isso, a função de custo vira um vale alongado e o algoritmo converge devagar. Modelos regularizados também precisam de variáveis padronizadas.
</details>

3. O gradiente descendente pode ficar preso num mínimo local ao treinar uma regressão logística?
<details><summary>Resposta</summary>
Não. A função de custo é convexa: tem um único mínimo (global).
</details>

4. O erro de validação sobe a cada época. O que pode estar acontecendo?
<details><summary>Resposta</summary>
Se o erro de treino também sobe, a taxa de aprendizado está alta demais e o algoritmo diverge: reduza <code>eta</code>. Se o de treino continua caindo, é overfitting: pare o treino (early stopping) ou regularize.
</details>

5. O erro de validação é muito maior que o de treino. Qual o diagnóstico e quais as saídas?
<details><summary>Resposta</summary>
Overfitting. Saídas: reduzir a complexidade (menos variáveis, grau menor), regularizar (aumentar alpha) ou coletar mais dados.
</details>

6. Treino e validação têm erros parecidos e ambos altos. Qual o diagnóstico?
<details><summary>Resposta</summary>
Underfitting (viés alto). Tente um modelo mais flexível ou reduza a regularização (alpha).
</details>

7. Quando preferir Ridge, Lasso ou Elastic Net?
<details><summary>Resposta</summary>
Ridge é um bom padrão. Lasso, se você acha que poucas variáveis importam (zera as outras). Elastic Net costuma ser mais seguro que a Lasso quando há variáveis correlacionadas ou mais variáveis que observações.
</details>

8. Quero classificar fotos como interno/externo e dia/noite. Devo usar softmax?
<details><summary>Resposta</summary>
Não. As classes não são mutuamente exclusivas (as quatro combinações ocorrem), então o problema é <em>multilabel</em>: treine dois classificadores de regressão logística. A softmax serve para classes exclusivas.
</details>

### Antes de sair
1. Um conceito que ficou claro.
2. Uma dúvida que ainda tem.
3. Um problema do seu trabalho em que você usaria regularização, e por quê.

### Exercícios extras (do capítulo 4)
1. **Early stopping na regressão logística:** na seção 6, tente variar `C` e `eta` e observe quando a parada antecipada acontece.
2. **Taxa de aprendizado:** na seção 2, implemente a busca de um bom `eta` testando uma lista de valores e comparando o erro depois de 100 épocas.
3. **Modelo de negócios:** pegue a base do Banco Mundial e preveja `expectativa_vida` com `log_pib`, `desemprego_pct` e `log_populacao` usando `Ridge`, `Lasso` e `ElasticNet` com `RidgeCV`/`LassoCV`/`ElasticNetCV`. Qual variável a Lasso elimina?
4. **Regressão logística em dados reais:** crie uma variável binária (por exemplo, "expectativa de vida acima de 75 anos") e treine uma `LogisticRegression`. Escolha o limite de probabilidade pelo custo dos erros, como na Monitoria 2.

### Referências
- Géron, A. *Hands-On Machine Learning with Scikit-Learn, Keras and TensorFlow* (3ª ed.) — [notebook](https://github.com/ageron/handson-ml3/blob/main/04_training_linear_models.ipynb): `04_training_linear_models.ipynb`.
- Guia do scikit-learn: [modelos lineares](https://scikit-learn.org/stable/modules/linear_model.html) e [curvas de aprendizado](https://scikit-learn.org/stable/modules/learning_curve.html).